# Log-PCA of Feedforward Inputs (`compute-log-pcas`)

Feedforward spike trains (1 500 mitral cells) are Gaussian-smoothed, log-transformed
(`log(rates + ε)`), and projected onto principal components.

Reconstruction back to rate space:
`rates ≈ exp(pc_timeseries[:, :k] @ components[:k] + mean) - log_epsilon`

Results are loaded across a grid search over smoothing width σ.

Sections:
1. **Cumulative explained variance** — one curve per σ (in log-rate space).
2. **Example reconstructions** — per σ, with varying numbers of PCs (displayed in Hz).
3. **R² vs number of PCs** — all σ values (R² in rate space).
4. **R² vs smoothing width** — at 20 PCs.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import zarr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.visualization import use_project_style

use_project_style()


def gaussian_smooth(spikes, sigma, dt=1.0):
    """Gaussian-smooth spike trains along the time axis on GPU."""
    width = int(6 * sigma)
    t = torch.arange(-width, width + 1, dtype=torch.float32)
    kernel = torch.exp(-(t**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()
    kernel = kernel.view(1, 1, -1).to(spikes.device)
    x = spikes.permute(0, 2, 1).reshape(-1, 1, spikes.shape[1])
    smoothed = F.conv1d(x, kernel, padding=width)
    return smoothed.reshape(spikes.shape[0], spikes.shape[2], spikes.shape[1]).permute(
        0, 2, 1
    )

In [ ]:
# ---------- Configuration ----------
sigma_values = [50, 100, 200, 500, 1000, 2000]  # ms
base_dir = load_experiment_config("experiment.toml")["output_dir"]

# ---------- Load all runs ----------
runs = {}  # sigma -> dict
for sigma in sigma_values:
    d = base_dir / f"sigma-{sigma}ms"
    z = zarr.open_group(d / "results" / "log_pca_inputs.zarr", mode="r")
    runs[sigma] = {
        "evr": z["explained_variance_ratio"][:],
        "pc_timeseries": z["pc_timeseries"],  # lazy
        "components": z["components"][:],
        "mean": z["mean"][:],
        "dt": float(z.attrs["dt"]),
        "n_components": int(z.attrs["n_components"]),
        "log_epsilon": float(z.attrs["log_epsilon"]),
        "run_dir": d,
    }
    print(f"σ={sigma:>5} ms : {z['pc_timeseries'].shape}")

# Shared constants from first run
n_ff = runs[sigma_values[0]]["components"].shape[1]
dt = runs[sigma_values[0]]["dt"]
hz_scale = 1000.0 / dt
log_epsilon = runs[sigma_values[0]]["log_epsilon"]

## Cumulative Explained Variance

Variance is computed in **log-rate space** — the space PCA is optimised in.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(sigma_values)))

for color, sigma in zip(cmap, sigma_values):
    evr = runs[sigma]["evr"]
    cumev = np.cumsum(evr)
    pc_idx = np.arange(1, len(evr) + 1)
    ax.plot(pc_idx, cumev * 100, color=color, lw=1.2, label=f"{sigma} ms")

for thresh, ls in [(90, "--"), (95, "-."), (99, ":")]:
    ax.axhline(thresh, color="gray", linestyle=ls, lw=0.7, alpha=0.5)

ax.set_xlabel("Number of PCs")
ax.set_ylabel("Cumulative Explained Variance (%)")
ax.set_title(
    f"Cumulative Explained Variance in Log-Rate Space  (ε = {log_epsilon})",
    fontweight="bold",
)
ax.legend(title="σ")
plt.tight_layout()
plt.show()

## Example Reconstructions

Smoothed rates (black) versus log-PCA reconstructions in **rate space** (Hz),
inverting the log transform: `exp(pcs @ components + mean) - ε`.

In [ ]:
rng = np.random.RandomState(42)
n_example = 6
example_nids = np.sort(rng.choice(n_ff, n_example, replace=False))
example_batch = 0
smooth_device = "cuda" if torch.cuda.is_available() else "cpu"

for sigma in sigma_values:
    rd = runs[sigma]
    n_components = rd["n_components"]
    eps = rd["log_epsilon"]

    # Load and smooth spikes
    spike_zarr = zarr.open_group(rd["run_dir"] / "inputs" / "spike_data.zarr", mode="r")
    input_spikes = spike_zarr["input_spikes"][example_batch]  # (T, n_ff)
    sigma_steps = sigma / dt
    spikes_t = (
        torch.from_numpy(input_spikes.astype(np.float32)).unsqueeze(0).to(smooth_device)
    )
    rates_smooth = gaussian_smooth(spikes_t, sigma=sigma_steps).cpu().numpy()[0]
    del spikes_t

    n_timesteps = rates_smooth.shape[0]
    t = np.arange(n_timesteps) * dt / 1000.0

    _ks = dict.fromkeys(k for k in [10, 30, 100, n_components] if k <= n_components)
    k_values_recon = list(_ks)
    colors_recon = ["C1", "C2", "C0", "C3"][: len(k_values_recon)]

    pcs = rd["pc_timeseries"][example_batch]  # (T, n_comp)

    fig, axes = plt.subplots(n_example, 1, figsize=(14, 2.5 * n_example), sharex=True)
    for i, nid in enumerate(example_nids):
        ax = axes[i]
        ax.plot(
            t,
            rates_smooth[:, nid] * hz_scale,
            "k-",
            lw=1.4,
            label="Smoothed",
            zorder=3,
        )
        for k, color in zip(k_values_recon, colors_recon):
            log_rec = pcs[:, :k] @ rd["components"][:k] + rd["mean"]
            rec_hz = (np.exp(log_rec[:, nid]) - eps) * hz_scale
            lw = 1.4 if k == n_components else 0.9
            ax.plot(t, rec_hz, "-", color=color, lw=lw, alpha=0.85, label=f"{k} PCs")
        ax.set_ylabel(f"Neuron {nid}\n(Hz)")
        ax.set_ylim(0, None)
        if i == 0:
            ax.legend(ncol=len(k_values_recon) + 1, fontsize=8)

    axes[-1].set_xlabel("Time (s)")
    plt.suptitle(
        f"Log-PCA Reconstructions  (trial {example_batch},  σ = {sigma} ms,  ε = {eps})",
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    if smooth_device == "cuda":
        torch.cuda.empty_cache()

## Reconstruction $R^2$ vs Number of PCs

$R^2$ computed in **rate space** (after inverting the log transform),
globally across all neurons, batches, and timesteps.

In [ ]:
k_values = [5, 10, 20, 30, 50, 100, 200, 500, 1000, 1500]
smooth_device = "cuda" if torch.cuda.is_available() else "cpu"

r2_results = {}  # sigma -> (k_arr, r2_arr)

for sigma in tqdm(sigma_values, desc="R² sweep"):
    rd = runs[sigma]
    eps = rd["log_epsilon"]
    k_valid = np.array([k for k in k_values if k <= rd["n_components"]])

    # Load and smooth spikes
    spike_zarr = zarr.open_group(rd["run_dir"] / "inputs" / "spike_data.zarr", mode="r")
    input_spikes = spike_zarr["input_spikes"][:]  # (B, T, n_ff)
    sigma_steps = sigma / dt
    spikes_t = torch.from_numpy(input_spikes.astype(np.float32)).to(smooth_device)
    rates_smooth = gaussian_smooth(spikes_t, sigma=sigma_steps).cpu().numpy()
    del spikes_t
    if smooth_device == "cuda":
        torch.cuda.empty_cache()

    comp_f32 = rd["components"].astype(np.float32)
    mean_f32 = rd["mean"].astype(np.float32)
    mean_global = rates_smooth.mean(axis=(0, 1))
    n_batches = rates_smooth.shape[0]

    ss_tot = 0.0
    ss_res = {k: 0.0 for k in k_valid}

    for b in range(n_batches):
        rates_b = rates_smooth[b].astype(np.float32)
        pcs_b = np.asarray(rd["pc_timeseries"][b], dtype=np.float32)
        ss_tot += float(np.sum((rates_b - mean_global) ** 2))
        for k in k_valid:
            log_rec_b = pcs_b[:, :k] @ comp_f32[:k] + mean_f32
            rec_b = np.exp(log_rec_b) - eps
            ss_res[k] += float(np.sum((rates_b - rec_b) ** 2))

    r2_arr = np.array([1.0 - ss_res[k] / ss_tot for k in k_valid])
    r2_results[sigma] = (k_valid, r2_arr)
    print(
        f"σ={sigma} ms done — R²@20PCs = {r2_arr[k_valid.tolist().index(20)] * 100:.1f}%"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(sigma_values)))

for color, sigma in zip(cmap, sigma_values):
    k_sweep, r2 = r2_results[sigma]
    ax.plot(
        k_sweep, r2 * 100, "o-", color=color, markersize=4, lw=1.2, label=f"{sigma} ms"
    )

ax.set_xlabel("Number of PCs")
ax.set_ylabel("$R^2$ in rate space (%)")
ax.set_xlim(0, None)
ax.set_ylim(0, 101)
ax.axhline(100, color="gray", linestyle="--", lw=0.8, alpha=0.5)
ax.legend(title="σ")
ax.set_title(
    f"Reconstruction R² vs Number of PCs  (ε = {log_epsilon})",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## R² vs Smoothing Width at 20 PCs

In [ ]:
k_fixed = 20

r2_by_sigma = []
for sigma in sigma_values:
    k_arr, r2 = r2_results[sigma]
    idx = np.argmin(np.abs(k_arr - k_fixed))
    r2_by_sigma.append(r2[idx] * 100)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sigma_values, r2_by_sigma, "o-", color="C1", markersize=7, lw=1.5)
ax.set_xlabel("Smoothing σ (ms)")
ax.set_xscale("log")
ax.set_xticks(sigma_values)
ax.set_xticklabels([str(s) for s in sigma_values])
ax.set_ylabel("$R^2$ in rate space (%)")
ax.set_ylim(0, 101)
ax.axhline(100, color="gray", linestyle="--", lw=0.8, alpha=0.5)
ax.set_title(
    f"Reconstruction R² vs Smoothing Width  ({k_fixed} PCs,  ε = {log_epsilon})",
    fontweight="bold",
)
plt.tight_layout()
plt.show()